In [1]:
!pip install pandas

In [8]:
import requests
import time
import json
import re

OLLAMA_MODEL = "qwen3.5:4b"

def call_llm(prompt, model=OLLAMA_MODEL, temperature=0.3, max_tokens=400):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "think": False,
                "options": {"temperature": temperature, "num_predict": max_tokens}
            },
            timeout=60
        )
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"LLM call failed: {e}"

print("Ollama LLM helper loaded")

Ollama LLM helper loaded


In [9]:
with open("../outputs/resume_text.txt", "r", encoding="utf-8") as f:
    resume_text = f.read()

print(f"Loaded resume text: {len(resume_text)} characters")

Loaded resume text: 2197 characters


In [10]:
TECHNICAL_SKILLS = [
    "Python", "Java", "JavaScript", "C++", "C", "SQL", "HTML", "CSS",
    "React", "Node.js", "FastAPI", "Django", "Flask", "MongoDB",
    "PostgreSQL", "MySQL", "AWS", "Azure", "Docker", "Kubernetes",
    "Git", "GitHub", "TypeScript", "Machine Learning", "Deep Learning",
    "AI", "Artificial Intelligence", "Data Science", "TensorFlow",
    "PyTorch", "NumPy", "Pandas", "LangChain", "REST API", "Linux",
    "DSA", "Data Structures", "Algorithms", "OOP", "DBMS"
]

SOFT_SKILLS = [
    "Leadership", "Communication", "Teamwork", "Problem Solving",
    "Collaboration", "Analytics", "Innovation", "Digital Marketing",
    "Time Management", "Critical Thinking", "Adaptability"
]

def extract_skills(text):
    text_lower = text.lower()
    technical_found = [s for s in TECHNICAL_SKILLS if s.lower() in text_lower]
    soft_found = [s for s in SOFT_SKILLS if s.lower() in text_lower]
    return {"technical": technical_found, "soft": soft_found}

def extract_education(text):
    education_entries = []
    lines = text.split("\n")
    degree_keywords = ["bachelor of", "b.sc.", "diploma in", "senior secondary education"]
    for line in lines:
        line_stripped = line.strip()
        line_lower = line_stripped.lower()
        if any(kw in line_lower for kw in degree_keywords) and len(line_stripped) < 60:
            education_entries.append(line_stripped)
    return education_entries

def extract_projects(text):
    projects = []
    text_upper = text.upper()
    if "PROJECTS" in text_upper:
        start = text_upper.find("PROJECTS") + len("PROJECTS")
        end = text_upper.find("\nSKILLS", start)
        if end == -1:
            end = len(text)
        section = text[start:end].strip()
        lines = [l.strip() for l in section.split("\n") if l.strip()]
        projects = lines
    return projects

regex_skills = extract_skills(resume_text)
regex_education = extract_education(resume_text)
regex_projects = extract_projects(resume_text)

regex_result = {
    "skills": regex_skills,
    "education": regex_education,
    "projects": regex_projects
}

print("Regex-Based Skill Extraction Agent: SUCCESS\n")
print(json.dumps(regex_result, indent=2))

Regex-Based Skill Extraction Agent: SUCCESS

{
  "skills": {
    "technical": [
      "Python",
      "Java",
      "JavaScript",
      "C++",
      "C",
      "SQL",
      "HTML",
      "CSS",
      "Flask",
      "Git",
      "Machine Learning",
      "AI",
      "Artificial Intelligence",
      "Data Science",
      "TensorFlow",
      "PyTorch",
      "NumPy",
      "Pandas",
      "Algorithms"
    ],
    "soft": [
      "Leadership",
      "Communication",
      "Teamwork",
      "Problem Solving",
      "Collaboration",
      "Analytics",
      "Innovation",
      "Digital Marketing"
    ]
  },
  "education": [
    "2024 Senior Secondary Education",
    "2024 - 2025 Diploma in Computer Application",
    "2024 - 2027 Bachelor of Science (AI & ML)"
  ],
  "projects": [
    "Personal Portfolio Web Developed a responsive Personal Portfolio Web Page using HTML, CSS, and",
    "Page: JavaScript to showcase skills, projects, certifications, and contact information with",
    "a clean an

In [11]:
def llm_extract_resume_data(resume_text):
    prompt = f"""Extract structured information from this resume. Return ONLY valid JSON, no other text, no markdown formatting, no explanation.

Format exactly like this:
{{
  "technical_skills": ["skill1", "skill2"],
  "soft_skills": ["skill1", "skill2"],
  "education": ["degree - institution - year"],
  "projects": [{{"name": "project name", "description": "one sentence summary"}}],
  "years_of_relevant_experience_estimate": "estimate based on projects/education, e.g. '0-1 years' or '1-2 years'"
}}

RESUME:
{resume_text[:2000]}

Return ONLY the JSON object, nothing else."""

    return call_llm(prompt, temperature=0.2, max_tokens=500)

start = time.time()
llm_extraction_raw = llm_extract_resume_data(resume_text)
elapsed = time.time() - start

print(f"LLM Extraction Agent: SUCCESS (generated in {elapsed:.2f}s)\n")
print(llm_extraction_raw)

LLM Extraction Agent: SUCCESS (generated in 6.94s)

{
  "technical_skills": [
    "Python",
    "Java",
    "C",
    "C++",
    "SQL",
    "HTML",
    "CSS",
    "JavaScript",
    "NumPy",
    "Pandas",
    "TensorFlow",
    "PyTorch",
    "Flask"
  ],
  "soft_skills": [
    "Leadership",
    "Digital Marketing",
    "Analytical Thinking",
    "Problem Solving",
    "Communication",
    "Innovation"
  ],
  "education": [
    "Senior Secondary Education - AIR FORCE SCHOOL AMBALA - 2024",
    "Diploma in Computer Application - HARTRON SKILL CENTRE AMBALA - 2025",
    "Bachelor of Science (AI & ML) - MAHARISHI MARKKANDESWAR UNIVERSITY - 2027"
  ],
  "projects": [
    {
      "name": "Personal Portfolio Web Page",
      "description": "Developed a responsive personal portfolio web page using HTML, CSS, and JavaScript to showcase skills, projects, certifications, and contact information."
    },
    {
      "name": "Predictive Supply Chain AI",
      "description": "Built an AI-based supply

In [13]:
def parse_llm_json(raw_text):
    try:
        return json.loads(raw_text)
    except:
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                return None
        return None

llm_extracted_data = parse_llm_json(llm_extraction_raw)

if llm_extracted_data:
    print("LLM output parsed successfully as valid JSON\n")
    print(json.dumps(llm_extracted_data, indent=2))
else:
    print("LLM output could not be parsed as JSON, falling back to regex-only extraction")
    llm_extracted_data = {}

LLM output parsed successfully as valid JSON

{
  "technical_skills": [
    "Python",
    "Java",
    "C",
    "C++",
    "SQL",
    "HTML",
    "CSS",
    "JavaScript",
    "NumPy",
    "Pandas",
    "TensorFlow",
    "PyTorch",
    "Flask"
  ],
  "soft_skills": [
    "Leadership",
    "Digital Marketing",
    "Analytical Thinking",
    "Problem Solving",
    "Communication",
    "Innovation"
  ],
  "education": [
    "Senior Secondary Education - AIR FORCE SCHOOL AMBALA - 2024",
    "Diploma in Computer Application - HARTRON SKILL CENTRE AMBALA - 2025",
    "Bachelor of Science (AI & ML) - MAHARISHI MARKKANDESWAR UNIVERSITY - 2027"
  ],
  "projects": [
    {
      "name": "Personal Portfolio Web Page",
      "description": "Developed a responsive personal portfolio web page using HTML, CSS, and JavaScript to showcase skills, projects, certifications, and contact information."
    },
    {
      "name": "Predictive Supply Chain AI",
      "description": "Built an AI-based supply chain

In [14]:
combined_result = {
    "regex_based": regex_result,
    "llm_based": llm_extracted_data,
    "llm_generation_time_seconds": round(elapsed, 2)
}

with open("../outputs/skills.json", "w", encoding="utf-8") as f:
    json.dump(combined_result, f, indent=2)

print("Combined skill extraction (regex + LLM) saved to ../outputs/skills.json")
print("Notebook 2 (Skill Extraction Agent) — UPGRADED WITH LLM — COMPLETE")

Combined skill extraction (regex + LLM) saved to ../outputs/skills.json
Notebook 2 (Skill Extraction Agent) — UPGRADED WITH LLM — COMPLETE
